In [5]:
from __future__ import annotations

import json
import os
from typing import Any

import psycopg
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv()

llm = ChatOpenAI(
    model="qwen3.8-flash",
    temperature=0,
    base_url="https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1",
    extra_body={"enable_thinking": False},
)


### Direct agent

In [6]:
def get_connection():
    """Connect as read-only sql_agent; password from SQL_AGENT_PASSWORD in .env."""
    return psycopg.connect(
        host=os.getenv("POSTGRES_HOST", "localhost"),
        port=int(os.getenv("POSTGRES_PORT", "5432")),
        dbname=os.getenv("POSTGRES_DB", "tilik"),
        user="sql_agent",
        password=os.environ["SQL_AGENT_PASSWORD"],
        options="-c default_transaction_read_only=on -c statement_timeout=10000",
    )

@tool
def list_tables() -> str:
    """List all tables in the database."""
    with get_connection() as conn:
        rows = conn.execute("SELECT table_name FROM information_schema.tables")
    return json.dumps([row[0] for row in rows])

@tool
def describe_table(table_name: str) -> str:
    """Get the schema of a table and the first 5 sample rows of data."""
    with get_connection() as conn:
        cols = conn.execute(f"SELECT column_name, data_type FROM information_schema.columns WHERE table_name = '{table_name}' AND table_schema = 'public'")
        if not cols:
            return f"Table {table_name} not found"
        rows = conn.execute(f"SELECT * FROM {table_name} LIMIT 5")
    return json.dumps({
        "columns": [col[0] for col in cols],
        "types": [col[1] for col in cols],
        "sample": [dict(zip([col[0] for col in cols], row)) for row in rows],
    })

@tool
def run_query(query: str) -> str:
    """Run a Read-Only Postgres SQL query on the database."""
    if not query.strip().lower().startswith("select"):
        return "Error: Only SELECT queries are supported"
    try:
        with get_connection() as conn:
            cursor = conn.execute(query)
            columns = [col[0] for col in cursor.description]
            rows = cursor.fetchall()
            return json.dumps({
                "columns": columns,
                "rows": rows,
            })
    except Exception as e:
        return f"Error running query: {e}"

tools = [list_tables, describe_table, run_query]

SYSTEM_PROMPT = SystemMessage(
    "You are a careful data analyst querying a star-schema warehouse.\n"
    "- Fact tables: fact_sales (grain: one row per order line; measures: quantity, revenue).\n"
    "- Dimensions: dim_customer, dim_product, dim_date. Join facts to dimensions on *_key columns only.\n"
    "- Never join two fact tables directly; aggregate each first, then combine.\n"
    "- Revenue = SUM(fact_sales.revenue). 'Last quarter' means fiscal quarter via dim_date.\n"
    "Inspect the schema before writing queries. Only use SELECT."
)

llm_with_tools = llm.bind_tools(tools)


def agent_node(state: MessagesState) -> dict[str, Any]:
    """Call the LLM; it may reply or request tool calls."""
    response = llm_with_tools.invoke([SYSTEM_PROMPT, *state["messages"]])
    return {"messages": [response]}


# --- Graph definition ---
graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")

sql_agent = graph.compile()

In [9]:
QUESTION = HumanMessage(
    content="Jelaskan Persentase Penduduk Berumur Kurang Dari Sama Dengan 18 Tahun yang Merokok Tembakau"
)

# Default LangGraph recursion_limit is 25 (agent <-> tools steps). Raise until tooling is tighter.
GRAPH_CONFIG = {"recursion_limit": 50}

step_messages: list[Any] = []
for step in sql_agent.stream(
    {"messages": [QUESTION]},
    config=GRAPH_CONFIG,
    stream_mode="updates",
):
    for node_name, update in step.items():
        print(f"\n=== node: {node_name} ===")
        for msg in update.get("messages", []):
            msg.pretty_print()
            step_messages.append(msg)

print("\n=== final answer ===")
print(step_messages[-1].content)


=== node: agent ===
================================== Ai Message ==================================

Saya akan memeriksa struktur database terlebih dahulu untuk memahami data yang tersedia.
Tool Calls:
  list_tables (call_10d63dc69a9b4f0f87a8d21d)
 Call ID: call_10d63dc69a9b4f0f87a8d21d
  Args:

=== node: tools ===
================================= Tool Message =================================
Name: list_tables

["langchain_pg_collection", "langchain_pg_embedding", "observation", "pg_type", "alembic_version", "pg_foreign_table", "label_mapping", "pg_roles", "geography", "dataset", "pg_settings", "ingestion_checkpoint", "ingestion_run", "raw_object", "pg_tables", "measure", "unit", "dimension", "period", "dimension_member", "pg_subscription", "pg_attribute", "pg_proc", "pg_class", "pg_attrdef", "pg_constraint", "pg_inherits", "pg_index", "pg_operator", "pg_statio_all_sequences", "pg_opfamily", "pg_opclass", "pg_am", "pg_amop", "pg_amproc", "pg_language", "pg_largeobject_metadata", "p